In [1]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

In [2]:
spark = SparkSession.builder \
.config("spark.warehouse.dir", f"/user/{username}/") \
.config('spark.ui.port','0') \
.enableHiveSupport() \
.master("yarn") \
.appName("reviews025320") \
.getOrCreate()

In [3]:
load_reviews_rdd = spark.sparkContext.textFile("datasets/reviews.csv")
load_reviews_rdd.take(2)

['business_name,author_name,text,photo,rating,rating_category',
 "Haci'nin Yeri - Yigit Lokantasi,Gulsum Akar,We went to Marmaris with my wife for a holiday. We chose this restaurant as a place for dinner based on the reviews and because we wanted juicy food. When we first went there was a serious queue. You proceed by taking the food you want in the form of an open buffet. Both vegetable dishes and meat dishes were plentiful. There was also dessert for those who wanted it. After you get what you want you pay at the cashier. They don't go through cards they work in cash. There was a lot of food variety. And the food prices were unbelievably cheap. We paid only 84 TL for all the meals here. It included buttermilk and bread. But unfortunately I can't say it's too clean as a place..,dataset/taste/hacinin_yeri_gulsum_akar.png,5,taste"]

In [4]:
boring_words_rdd = spark.sparkContext.textFile("datasets/boringwords.txt")
boring_words_rdd.take(5)

['shouldnt', 'worrying', 'simplify', 'tidy', 'shouldnt']

In [5]:
#load_reviews_rdd.map(lambda x: x.split(",")).filter(lambda)

In [6]:
review_text_rdd = load_reviews_rdd.map(lambda x: x.split(",")[2])
review_text_rdd.take(5)

['text',
 "We went to Marmaris with my wife for a holiday. We chose this restaurant as a place for dinner based on the reviews and because we wanted juicy food. When we first went there was a serious queue. You proceed by taking the food you want in the form of an open buffet. Both vegetable dishes and meat dishes were plentiful. There was also dessert for those who wanted it. After you get what you want you pay at the cashier. They don't go through cards they work in cash. There was a lot of food variety. And the food prices were unbelievably cheap. We paid only 84 TL for all the meals here. It included buttermilk and bread. But unfortunately I can't say it's too clean as a place..",
 "During my holiday in Marmaris we ate here to fit the food. It's really good that the food is cheap and nice. Eating as much bread as you want is a big plus for those who are not satisfied without bread. It is a place that I can recommend to those who will go to Marmaris. On July 1 there was a small incr

In [7]:
review_text_rdd.count()

1101

In [8]:
boring_words_rdd.count()

10461

In [9]:
boring_words_set = set(boring_words_rdd.collect())
broadcast_boring_words = spark.sparkContext.broadcast(boring_words_set)

In [10]:
import re
def remove_boring_words(text):
    # words = text.lower().split(" ")
    words = re.findall(r'\b\w+\b', text.lower())
    return [w for w in words if w not in broadcast_boring_words.value]

In [11]:
cleaned_reviews_text_rdd = review_text_rdd.flatMap(remove_boring_words)

In [13]:
cleaned_reviews_text_rdd.take(5)

['marmaris', 'juicy', 'buffet', 'plentiful', 'dessert']

In [15]:
agg_review_word_count = cleaned_reviews_text_rdd.map(lambda x: (x, 1)).reduceByKey(lambda x,y: x+y).sortBy(lambda x: x[1], False)

In [17]:
agg_review_word_count.take(20)

[('waiters', 48),
 ('tl', 43),
 ('ambiance', 38),
 ('tasty', 38),
 ('flavors', 37),
 ('didn', 34),
 ('appetizers', 32),
 ('lahmacun', 32),
 ('crowded', 28),
 ('tasted', 26),
 ('treats', 26),
 ('kebab', 25),
 ('doner', 23),
 ('dessert', 23),
 ('hamburger', 22),
 ('meatballs', 22),
 ('1', 21),
 ('burger', 21),
 ('tastes', 20),
 ('dough', 20)]